# ЛР-04: Назначение аварийных бригад на районы

## Worked example: civil 01

Это полностью разобранный пример по модели назначений: от постановки до интерпретации и анализа мини-сценария.

## 1. Исходный кейс

Городской диспетчерский центр планирует назначение аварийных бригад на районы на ближайшую смену.

Критерий оптимизации: **время реагирования, мин** (минимизация).

| Исполнитель \ Задача | Район-1 | Район-2 | Район-3 | Район-4 |
| --- | --- | --- | --- | --- |
| Бригада Север | 16 | 12 | 15 | 14 |
| Бригада Юг | 14 | 15 | 13 | 12 |
| Бригада Запад | 13 | 14 | 12 | 16 |
| Бригада Центр | 15 | 13 | 14 | 11 |


## Шаг 1. Подготовка данных

На этом этапе мы задаём данные кейса и функции-помощники.
Ожидаемый результат шага: готовая функция решения и корректные входные структуры.

In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment, linprog

def solve_assignment(cost_matrix):
    row_ind, col_ind = linear_sum_assignment(cost_matrix)
    total_cost = float(cost_matrix[row_ind, col_ind].sum())
    return row_ind, col_ind, total_cost

def assignment_table(worker_names, task_names, cost_matrix, row_ind, col_ind):
    rows = []
    for i, j in zip(row_ind, col_ind):
        rows.append({
            'исполнитель': worker_names[i],
            'задача': task_names[j],
            'затраты': float(cost_matrix[i, j]),
        })
    return pd.DataFrame(rows).sort_values('исполнитель').reset_index(drop=True)

def solve_assignment_via_lp(cost_matrix):
    n_rows, n_cols = cost_matrix.shape
    if n_rows != n_cols:
        raise ValueError('Ожидается квадратная матрица затрат.')
    n = n_rows
    c = cost_matrix.flatten()
    A_eq = []
    b_eq = []

    for i in range(n):
        row = np.zeros(n * n)
        row[i * n:(i + 1) * n] = 1.0
        A_eq.append(row)
        b_eq.append(1.0)

    for j in range(n):
        col = np.zeros(n * n)
        col[j::n] = 1.0
        A_eq.append(col)
        b_eq.append(1.0)

    result = linprog(
        c,
        A_eq=np.array(A_eq),
        b_eq=np.array(b_eq),
        bounds=[(0, 1)] * (n * n),
        method='highs',
    )
    if not result.success:
        raise RuntimeError(result.message)
    assignment_matrix = result.x.reshape(n, n)
    return result, assignment_matrix

worker_names = ['Бригада Север', 'Бригада Юг', 'Бригада Запад', 'Бригада Центр']
task_names = ['Район-1', 'Район-2', 'Район-3', 'Район-4']
cost_matrix = np.array([[16, 12, 15, 14], [14, 15, 13, 12], [13, 14, 12, 16], [15, 13, 14, 11]], dtype=float)

## Шаг 2. Базовое решение

Получаем оптимальное назначение и суммарный критерий.
Ожидаемый результат шага: `row_ind`, `col_ind`, `total_cost`.

In [2]:
row_ind, col_ind, total_cost = solve_assignment(cost_matrix)
assignment_df = assignment_table(worker_names, task_names, cost_matrix, row_ind, col_ind)
print('Оптимальный суммарный критерий (метод Венгера):', round(total_cost, 2))

Оптимальный суммарный критерий (метод Венгера): 49.0


## Шаг 3. Табличный результат

Показываем итоговое назначение и проверяем «один к одному» ограничения.
Ожидаемый результат шага: корректная таблица `assignment_df` и пройденные `assert`-проверки.

In [3]:
assert len(set(row_ind)) == len(worker_names), 'Исполнитель повторяется.'
assert len(set(col_ind)) == len(task_names), 'Задача повторяется.'
assert assignment_df['исполнитель'].nunique() == len(worker_names), 'Дубликаты исполнителей в таблице.'
assert assignment_df['задача'].nunique() == len(task_names), 'Дубликаты задач в таблице.'

display(assignment_df)

,исполнитель,задача,затраты
0,Бригада Запад,Район-3,12.0
1,Бригада Север,Район-2,12.0
2,Бригада Центр,Район-4,11.0
3,Бригада Юг,Район-1,14.0


## Шаг 4. LP-проверка

Сверяем решение методом Венгера с LP-постановкой.
Ожидаемый результат шага: таблица сравнения критериев двух подходов.

In [4]:
lp_result, lp_matrix = solve_assignment_via_lp(cost_matrix)
lp_total = float(lp_result.fun)
comparison_df = pd.DataFrame({
    'метод': ['Метод Венгера', 'LP-проверка (linprog)'],
    'суммарный критерий': [round(total_cost, 4), round(lp_total, 4)],
})
display(comparison_df)
display(pd.DataFrame(np.round(lp_matrix, 3), index=worker_names, columns=task_names))

,метод,суммарный критерий
0,Метод Венгера,49.0
1,LP-проверка (linprog),49.0


,Район-1,Район-2,Район-3,Район-4
Бригада Север,-0.0,1.0,0.0,0.0
Бригада Юг,1.0,0.0,0.0,0.0
Бригада Запад,0.0,0.0,1.0,0.0
Бригада Центр,-0.0,0.0,-0.0,1.0


## Шаг 5. Мини-сценарий изменений

Меняем одну стоимость и смотрим, как это влияет на итог.
Ожидаемый результат шага: таблица с базовым и новым значением критерия.

In [5]:
# Сценарий: Рост времени в связке Бригада Север -> Район-2 (+3)
scenario_matrix = cost_matrix.copy()
scenario_matrix[0, 1] += 3.0
_, _, scenario_total = solve_assignment(scenario_matrix)
delta = scenario_total - total_cost

scenario_df = pd.DataFrame({
    'сценарий': ['Рост времени в связке Бригада Север -> Район-2 (+3)'],
    'базовый критерий': [round(total_cost, 2)],
    'новый критерий': [round(scenario_total, 2)],
    'изменение': [round(delta, 2)],
})
display(scenario_df)

,сценарий,базовый критерий,новый критерий,изменение
0,Рост времени в связке Бригада Север -> Район-2...,49.0,52.0,3.0


## Шаг 6. Итоговый учебный вывод

Что студент должен вынести из примера:

- как поэтапно строится решение задачи о назначениях;
- как проверять корректность «один к одному» ограничений;
- как использовать LP-проверку как независимую валидацию;
- как интерпретировать изменение решения при точечной корректировке параметров.

Для отчёта важно проговорить не только числа, но и содержательный смысл выбранного назначения.